In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Qiskit Aer primitiveによる完全シミュレーションとノイズありシミュレーション

<details>
<summary><b>Package versions</b></summary>

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
```
</details>

[Qiskit primitiveによる完全シミュレーション](/guides/simulate-with-qiskit-sdk-primitives)では、Qiskitに含まれるリファレンスprimitiveを使用して量子回路の完全シミュレーションを実行する方法を説明しています。現在の量子プロセッサーにはエラー（ノイズ）が存在するため、完全シミュレーションの結果が実際のハードウェア上で回路を実行したときに期待される結果と必ずしも一致するわけではありません。Qiskitのリファレンスprimitiveはノイズのモデリングをサポートしていませんが、[Qiskit Aer](https://qiskit.org/ecosystem/aer/)にはノイズモデリングをサポートするprimitiveの実装が含まれています。Qiskit Aerは高性能な量子回路シミュレーターであり、より高いパフォーマンスと多くの機能を実現するためにリファレンスprimitiveの代わりに使用できます。Qiskit Aerは[Qiskit Ecosystem](https://qiskit.github.io/ecosystem/)の一部です。この記事では、完全シミュレーションとノイズありシミュレーションにおけるQiskit Aer primitiveの使用方法を説明します。

> **Note:** - `qiskit-aer` v0.14以降が必要です。
> - Qiskit Aer primitiveはprimitive インターフェースを実装していますが、Qiskit Runtime primitiveと同じオプションを提供するわけではありません。たとえば、耐性レベル（Resilience level）はQiskit Aer primitiveでは利用できません。
> - Aerがサポートするシミュレーション方式のオプションについては、[AerSimulatorのドキュメント](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator)を参照してください。

完全シミュレーションとノイズありシミュレーションを試すために、8量子ビットのサンプル回路を作成します。

In [1]:
from qiskit.circuit.library import efficient_su2

n_qubits = 8
circuit = efficient_su2(n_qubits)
circuit.draw("mpl")

<Image src="../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/simulate-with-qiskit-aer/extracted-outputs/df70b5fd-971d-4e7d-a23a-8df037c0fa47-0.svg)

この回路には、$R_y$ゲートと$R_z$ゲートの回転角を表すパラメーターが含まれています。この回路をシミュレーションする際には、これらのパラメーターに具体的な値を指定する必要があります。次のセルでは、これらのパラメーターにいくつかの値を指定し、Qiskit AerのEstimator primitiveを使用して、観測量$ZZ \cdots Z$の厳密な期待値を計算します。

In [2]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager
from qiskit_aer import AerSimulator
from qiskit_aer.primitives import EstimatorV2 as Estimator

observable = SparsePauliOp("Z" * n_qubits)
params = [0.1] * circuit.num_parameters

exact_estimator = Estimator()
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(circuit)
pub = (isa_circuit, observable, params)
job = exact_estimator.run([pub])
result = job.result()
pub_result = result[0]
exact_value = float(pub_result.data.evs)
exact_value

0.8870140234256602

Now, let's initialize a noise model that includes depolarizing error of 2% on every CX gate. In practice, the error arising from the two-qubit gates, which are CX gates here, are the dominant source of error when running a circuit. See [Build noise models](./build-noise-models) for an overview of constructing noise models in Qiskit Aer.

In the next cell, we construct an Estimator that incorporates this noise model and use it to compute the expectation value of the observable.

In [3]:
from qiskit_aer.noise import NoiseModel, depolarizing_error

noise_model = NoiseModel()
cx_depolarizing_prob = 0.02
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(cx_depolarizing_prob, 2), ["cx"]
)

noisy_estimator = Estimator(
    options=dict(backend_options=dict(noise_model=noise_model))
)
job = noisy_estimator.run([pub])
result = job.result()
pub_result = result[0]
noisy_value = float(pub_result.data.evs)
noisy_value

0.7247404214143528

次に、すべてのCXゲートに2%の脱分極エラー（depolarizing error）を含むノイズモデルを初期化します。実際には、ここでのCXゲートのような2量子ビットゲートから生じるエラーが、回路を実行する際の主要なエラーの原因です。Qiskit Aerでのノイズモデルの構築概要については、[ノイズモデルの構築](./build-noise-models)を参照してください。

次のセルでは、このノイズモデルを組み込んだEstimatorを構築し、観測量の期待値を計算します。

In [4]:
cx_count = circuit.count_ops()["cx"]
(1 - cx_depolarizing_prob) ** cx_count

0.6542558123199923

This value, 65%, gives a rough estimate of the probability that our final state is correct. It is a conservative estimate because it does not take into account the initial state of the simulation.

The following code cell shows how to use the Sampler primitive from Qiskit Aer to sample from the noisy circuit. We need to add measurements to the circuit before running it with the Sampler primitive.

In [5]:
from qiskit_aer.primitives import SamplerV2 as Sampler

measured_circuit = circuit.copy()
measured_circuit.measure_all()

noisy_sampler = Sampler(
    options=dict(backend_options=dict(noise_model=noise_model))
)
# The circuit needs to be transpiled to the AerSimulator target
pass_manager = generate_preset_pass_manager(3, AerSimulator())
isa_circuit = pass_manager.run(measured_circuit)
pub = (isa_circuit, params, 100)
job = noisy_sampler.run([pub])
result = job.result()
pub_result = result[0]
pub_result.data.meas.get_counts()

{'00100000': 1,
 '00000000': 65,
 '10101000': 1,
 '10000000': 5,
 '00001000': 1,
 '00000110': 2,
 '11110010': 1,
 '00000011': 3,
 '01010000': 3,
 '11000000': 3,
 '01111000': 1,
 '01000000': 2,
 '00000010': 1,
 '01100000': 1,
 '00011000': 1,
 '00111100': 1,
 '00010100': 1,
 '00001111': 1,
 '00110000': 1,
 '01100101': 1,
 '00000100': 1,
 '10100000': 1,
 '00000001': 1,
 '11010000': 1}

ご覧のとおり、ノイズが存在する場合の期待値は正しい値から大きくかけ離れています。実際には、ノイズの影響を軽減するためにさまざまなエラー軽減手法を使用できますが、これらの手法の説明はこの記事の範囲外となります。

ノイズが最終結果にどのような影響を与えるかをおおまかに把握するために、各CXゲートに2%の脱分極エラーを追加するノイズモデルを考えてみましょう。確率$p$の脱分極エラーは、密度行列$\rho$に対して次の作用を持つ量子チャンネル$E$として定義されます。

$$
E(\rho) = (1 - p) \rho + p\frac{I}{2^n}
$$

ここで$n$は量子ビット数で、この場合は2です。つまり、確率$p$で状態が完全混合状態に置き換えられ、確率$1 - p$で状態が保存されます。脱分極チャンネルを$m$回適用した後、状態が保存される確率は$(1 - p)^m$となります。したがって、シミュレーションの終了時に正しい状態を保持する確率は、回路内のCXゲートの数に対して指数的に減少することが予想されます。

回路内のCXゲートの数を数えて$(1 - p)^m$を計算してみましょう。`count_ops`を呼び出してゲート名とカウントのマッピングを含む辞書を取得し、CXゲートのエントリーを取り出します。